# 정적 웹페이지 수집하기
* 순수 HTML, CSS로 만들어진 페이지
* javascript로 내용을 갱신하지 않는 페이지

# yes24 베스트셀러 자료 수집하기

In [3]:
import time
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd

In [4]:
url = "https://www.yes24.com/product/category/bestseller"
payload = dict(categoryNumber="001", pageNumber=1, pageSize=120)
r = requests.get(url, params=payload)
print(r.url)
print(r.status_code)
soup = bs(r.content, "lxml")
time.sleep(0.5)
# soup

https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=1&pageSize=120
200


* 전체 yes24 베스트셀러 페이지 중 책 정보가 들어있는 곳: ul#yesBestList
* ul#yesBestList 아래의 li에 책 1권의 정보가 들어있음

In [5]:
# yes24 전체 페이지에서 책 정보가 들어있는 부분만 잘라서 book_list에 저장
book_list = soup.select("ul#yesBestList > li")

In [6]:
book_list

[<li class="" data-goods-no="144021119" data-iy-no="0" data-statgb="02">
 <div class="itemUnit">
 <div class="item_img">
 <div class="img_canvas">
 <div class="img_upper">
 <em class="ico rank">1</em>
 <span class="rank_info rank_even">
 <span class="ico"></span><em class="txt rank"></em><em class="txt">위 상승</em>
 </span>
 </div>
 <span class="img_item">
 <span class="img_grp">
 <a class="lnk_img" href="/product/goods/144021119" onclick="wiseLogV2('BS', '001_005_001', ''); ">
 <em class="img_bdr">
 <img alt="단 한 번의 삶" border="0" class="lazy" data-original="https://image.yes24.com/goods/144021119/L" src="https://image.yes24.com/momo/Noimg_L.jpg"/>
 </em>
 </a>
 </span>
 </span>
 </div>
 <div class="img_btn">
 <a class="btnC btn_preview" href="javascript:yes24GU.openPreviewCheck(144021119); wiseLogV2('BS', '001_005_011', '');"><span class="bWrap"><em class="txt">미리보기</em></span></a>
 </div>
 </div>
 <div class="item_info">
 <div class="info_row info_keynote">
 </div>
 <div class="info_ro

In [4]:
# for book in book_list:
#     print(book)
#     print()

In [4]:
result = {}
for idx, book in enumerate(book_list):
    print(f"{idx}/{len(book_list)} 추출 중", end="\r")
    # 책 제목
    book_title = book.select_one(".gd_name").text
    # 저자
    author = book.select_one(".info_row.info_pubGrp a").text
    # 출판사
    publisher = book.select_one(".authPub.info_pub a").text
    # 출간일
    date_pub = book.select_one(".authPub.info_date").text
    # 가격
    price = book.select_one(".info_row.info_price em.yes_b").text.replace(",", "")
    # 평점
    rating = book.select_one(".rating_grade em.yes_b").text if book.select_one(".rating_grade em.yes_b") != None else 0.0  
    # 리뷰 수
    n_reviews = book.select_one(".info_row.info_rating em.txC_blue").text if book.select_one(".info_row.info_rating em.txC_blue") != None else 0  
    
    keys = ["book_title", "author", "publisher", "date_pub", "price", "rating", "n_reviews"]
    values = [book_title, author, publisher, date_pub, price, rating, n_reviews]
    for key, value in zip(keys, values):
        result.setdefault(key, []).append(value)
    
for key, value in result.items():
    print(key, len(value))
    
df = pd.DataFrame(result)
df

book_title 120
author 120
publisher 120
date_pub 120
price 120
rating 120
n_reviews 120


,book_title,author,publisher,date_pub,price,rating,n_reviews
0,단 한 번의 삶,김영하,복복서가,2025년 04월,15120,8.7,14
1,듀얼 브레인,이선 몰릭,상상스퀘어,2025년 03월,18900,8.7,38
2,어른의 품격을 채우는 100일 필사 노트,김종원,청림Life,2025년 03월,18000,9.9,56
3,소년이 온다,한강,창비,2014년 05월,13500,9.7,"3,848"
4,성적 초격차를 만드는 독서력 수업,김수미,빅피시,2025년 03월,17820,9.9,69
...,...,...,...,...,...,...,...
115,해커스 토익 RC Reading(리딩) 기본서,David Cho,해커스어학연구소,2023년 07월,16920,9.8,241
116,침묵의 퍼레이드,히가시노 게이고,재인,2025년 03월,19620,10.0,10
117,2025 시대에듀 투자자산운용사 실제유형 모의고사 + 특별부록 PASSCODE Pr...,유창호,시대고시기획 시대교육,2025년 03월,49500,10.0,2
118,마흔 고비에 꼭 만나야 할 장자,이길환,이든서재,2025년 04월,16920,0.0,0


In [5]:
print("가격:", len(list(soup.select("ul#yesBestList .info_row.info_price em.yes_b"))))
print("평점:", len(list(soup.select("ul#yesBestList .rating_grade em.yes_b"))))
print("리뷰 수:", len(list(soup.select("ul#yesBestList .info_row.info_rating em.txC_blue"))))

가격: 120
평점: 106
리뷰 수: 106


In [6]:
result['author']

['김영하',
 '이선 몰릭',
 '김종원',
 '한강',
 '김수미',
 '태수',
 '와야마 야마',
 '코이케 류노스케',
 '요시타케 신스케',
 '양귀자',
 '김주환',
 '유발 하라리',
 '유선경',
 '김주환',
 '최태성',
 '한강',
 '최태성',
 '안녕달',
 '백온유',
 '정대건',
 '브라이언 트레이시',
 '성해나',
 'ETS',
 '존 윌리엄스',
 'ETS',
 '오건영',
 '한강',
 '스즈키 유우토',
 '한현근',
 '정관 스님',
 '김지훈(포메뽀꼬)',
 '프리드리히 니체',
 '안-엘렌 클레르',
 '김직선',
 '한강',
 '김종원',
 '최태성',
 'David Cho',
 '김종원',
 '루리',
 '김종원',
 '천근아',
 '벤저민 하디',
 '김종원',
 '유설화',
 '이철희',
 '김지훤',
 '유은하',
 '박준',
 '네오쇼코',
 '네오쇼코',
 '신재은',
 '위혜정',
 '흔한남매',
 '태 킴',
 '타치바나 오레코',
 '아이다이로',
 '이어령',
 '알에스미디어',
 '백수린',
 '최승필',
 '한강',
 '클레어 키건',
 '정수봉',
 '해커스 일본어연구소',
 '한동훈',
 '손원평',
 '네빌 고다드',
 '세이노(SayNo)',
 '칼 에드워드 세이건',
 '유현준',
 '강용수',
 '타츠 유키노부',
 '강보라',
 '니이 사토루',
 '김종원',
 '조수용',
 '정주희',
 '문상훈',
 '고시넷 NCS 연구소',
 '김화요',
 '오창석',
 '유발 하라리',
 '리처드 도킨스',
 '최승호 외',
 '데일 카네기',
 '제시카 윤',
 '박태웅',
 '인이이',
 '하마지 아키',
 '최설희',
 '김민철',
 '이꽃님',
 '박진여',
 '김주환',
 '센바 쿠로노',
 '헤르만 헤세',
 '한강',
 '백유연',
 '하마지 아키',
 '모건 하우절',
 '엄예정',
 '강효미',
 '도야마 시게히코',
 '혼Job취업연구소',
 '김종우',
 '최진영',


In [7]:
result['author'][29]

'정관 스님'

In [8]:
# 저자 수
len(book_list[29].select(".authPub.info_auth a"))

4

In [9]:
book_list[29].select_one(".authPub.info_auth").text.split("/")[0]

'\n정관 스님, 후남 셀만 저'

In [10]:
book_list[29].select_one(".authPub.info_auth").text.split("/")[1]

'베로니크 회거 사진'

In [11]:
book_list[29].select_one(".authPub.info_auth").text.split("/")[2]

'양혜영 역\r\n                            '

In [12]:
book_list[18].select_one(".authPub.info_auth").text.split("/")[1].replace("\n", " ").strip("감추기 ")

'백온유 강보라 서장원 성해나 성혜령 이희주 현호정'

In [13]:
# 상세 페이지 링크
link = "https://www.yes24.com" + book_list[20].select_one(".gd_name")['href']
link

'https://www.yes24.com/product/goods/138282792'

# 전체 페이지 수집하기

* for 반복문

In [14]:
result_list = []
for page in range(1, 10):
    url = "https://www.yes24.com/product/category/bestseller"
    payload = dict(categoryNumber="001", pageNumber=page, pageSize=120)
    r = requests.get(url, params=payload)
    print(r.url)
    print(r.status_code)
    soup = bs(r.content, "lxml")
    time.sleep(0.5)
    
    book_list = soup.select("ul#yesBestList > li")
    result = {}
    for idx, book in enumerate(book_list):
        print(f"{idx}/{len(book_list)} 추출 중", end="\r")
        # 책 제목
        book_title = book.select_one(".gd_name").text
        # 저자
        author = book.select_one(".info_row.info_pubGrp a").text
        # 출판사
        publisher = book.select_one(".authPub.info_pub a").text
        # 출간일
        date_pub = book.select_one(".authPub.info_date").text
        # 가격
        price = book.select_one(".info_row.info_price em.yes_b").text.replace(",", "")
        # 평점
        rating = book.select_one(".rating_grade em.yes_b").text if book.select_one(".rating_grade em.yes_b") != None else 0.0  
        # 리뷰 수
        n_reviews = book.select_one(".info_row.info_rating em.txC_blue").text if book.select_one(".info_row.info_rating em.txC_blue") != None else 0  

        keys = ["book_title", "author", "publisher", "date_pub", "price", "rating", "n_reviews"]
        values = [book_title, author, publisher, date_pub, price, rating, n_reviews]
        
        for key, value in zip(keys, values):
            result.setdefault(key, []).append(value)
            
    result_list.append(pd.DataFrame(result))

result_list = pd.concat(result_list)
result_list = result_list.reset_index(drop=True)
result_list

https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=1&pageSize=120
200
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=2&pageSize=120
200
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=3&pageSize=120
200
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=4&pageSize=120
200
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=5&pageSize=120
200
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=6&pageSize=120
200
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=7&pageSize=120
200
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=8&pageSize=120
200
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=9&pageSize=120
200


,book_title,author,publisher,date_pub,price,rating,n_reviews
0,단 한 번의 삶,김영하,복복서가,2025년 04월,15120,8.7,14
1,듀얼 브레인,이선 몰릭,상상스퀘어,2025년 03월,18900,8.7,38
2,어른의 품격을 채우는 100일 필사 노트,김종원,청림Life,2025년 03월,18000,9.9,56
3,소년이 온다,한강,창비,2014년 05월,13500,9.7,"3,848"
4,성적 초격차를 만드는 독서력 수업,김수미,빅피시,2025년 03월,17820,9.9,69
...,...,...,...,...,...,...,...
994,태도에 관하여 (20만 부 기념 완결판),임경선,토스트,2024년 09월,16200,9.3,33
995,장수탕 선녀님,백희나,스토리보울,2024년 04월,13500,9.8,24
996,중등 필독 신문,이현옥,체인지업,2024년 02월,16020,9.9,156
997,[예스리커버] 나는 나의 스무 살을 가장 존중한다,이하영,토네이도,2024년 08월,16200,9.4,212


* while 반복문

In [15]:
result_list = []
page = 1
while True:
    url = "https://www.yes24.com/product/category/bestseller"
    payload = dict(categoryNumber="001", pageNumber=page, pageSize=120)
    r = requests.get(url, params=payload)
    print(r.url)
    print(r.status_code)
    soup = bs(r.content, "lxml")
    time.sleep(5)
    # yes24 전체 페이지에서 책 정보가 들어있는 부분만 잘라서 book_list에 저장
    book_list = soup.select("ul#yesBestList > li")
    result = {}
    for idx, book in enumerate(book_list):
        print(f"{idx}/{len(book_list)} 추출중", end="\r")
        # 책 제목
        book_title = book.select_one(".gd_name").text
        # 저자
        author = book.select_one(".info_row.info_pubGrp a").text
        # 출판사
        publisher = book.select_one(".authPub.info_pub a").text
        # 출간일
        date_pub = book.select_one(".authPub.info_date").text
        # 가격
        price = book.select_one(".info_row.info_price em.yes_b").text
        # 평점
        rating = book.select_one(".rating_grade em.yes_b").text if book.select_one(".rating_grade em.yes_b") != None else 0.0
        # 리뷰수
        n_reviews = book.select_one(".info_row.info_rating em.txC_blue").text if book.select_one(".info_row.info_rating em.txC_blue") != None else 0

        keys = ['book_title', 'author', 'publisher', 'date_pub', 'price', 'rating', 'n_reviews']
        values = [book_title, author, publisher, date_pub, price, rating, n_reviews]
        for key, value in zip(keys, values):
            result.setdefault(key, []).append(value)

    for key, value in result.items():
        print(key, len(value))

    result_list.append(pd.DataFrame(result))
    
    if page < 12:
        page += 1
    else:
        break
    
result_list = pd.concat(result_list)
result_list = result_list.reset_index(drop=True)
result_list

https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=1&pageSize=120
200
book_title 120
author 120
publisher 120
date_pub 120
price 120
rating 120
n_reviews 120
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=2&pageSize=120
200
book_title 120
author 120
publisher 120
date_pub 120
price 120
rating 120
n_reviews 120
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=3&pageSize=120
200
book_title 120
author 120
publisher 120
date_pub 120
price 120
rating 120
n_reviews 120
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=4&pageSize=120
200
book_title 120
author 120
publisher 120
date_pub 120
price 120
rating 120
n_reviews 120
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=5&pageSize=120
200
book_title 120
author 120
publisher 120
date_pub 120
price 120
rating 120
n_reviews 120
https://www.yes24.com/product/category/bestseller?categoryNumber=

,book_title,author,publisher,date_pub,price,rating,n_reviews
0,단 한 번의 삶,김영하,복복서가,2025년 04월,"15,120",8.7,14
1,듀얼 브레인,이선 몰릭,상상스퀘어,2025년 03월,"18,900",8.7,38
2,어른의 품격을 채우는 100일 필사 노트,김종원,청림Life,2025년 03월,"18,000",9.9,56
3,소년이 온다,한강,창비,2014년 05월,"13,500",9.7,"3,848"
4,성적 초격차를 만드는 독서력 수업,김수미,빅피시,2025년 03월,"17,820",9.9,69
...,...,...,...,...,...,...,...
994,태도에 관하여 (20만 부 기념 완결판),임경선,토스트,2024년 09월,"16,200",9.3,33
995,장수탕 선녀님,백희나,스토리보울,2024년 04월,"13,500",9.8,24
996,중등 필독 신문,이현옥,체인지업,2024년 02월,"16,020",9.9,156
997,[예스리커버] 나는 나의 스무 살을 가장 존중한다,이하영,토네이도,2024년 08월,"16,200",9.4,212


# 저자, 역자, 글그림, 편저, 공동저자 등 구분하기

In [17]:
def text_clean(text):
    return text.replace("\n", " ").replace("\r", " ").strip()

In [28]:
def author_extraction(book):
    author = ""
    photo = ""
    trans = ""
    paint= ""
    for idx, item in enumerate(text_clean(book.select_one(".authPub.info_auth").text).split("/")):
        if idx == 0:
            if "저" == item[-1]:
                author = text_clean(item[:-2])
            elif "글" == item[-1]:
                author = text_clean(item[:-2])
            elif "글그림" == item[-3:]:
                author = text_clean(item[:-4])
        else:        
            if "정보 더 보기" == item[-7:]:
                author = text_clean(item.replace("정보 더 보기", ""))
            elif "사진" == item[-2:]:
                photo = text_clean(item[:-3])
            elif "역" == item[-1]:
                trans = text_clean(item[:-2])
            elif "글그림" == item[-3:]:
                paint = text_clean(item[:-4])
                
    print(f"author{author}, photo{photo}, trans{trans}, paint{paint}")
    return author, photo, trans, paint

In [29]:
url = "https://www.yes24.com/product/category/bestseller"
payload = dict(categoryNumber="001", pageNumber=1, pageSize=120)
r = requests.get(url, params=payload)
print(r.url)
print(r.status_code)
soup = bs(r.content, "lxml")
time.sleep(0.5)
book_list = soup.select("ul#yesBestList > li")
for book in book_list[:2]:
    author, photo, trans, paint = author_extraction(book)
    print(author, photo, trans, paint)

https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=1&pageSize=120
200
author김영하, photo, trans, paint
김영하   
author이선 몰릭, photo, trans신동숙, paint
이선 몰릭  신동숙 


In [22]:
for item in text_clean(book_list[18].select_one(".authPub.info_auth").text).split("/"):
#     print(item)
    if "정보 더 보기" == item[-7:] :
        author = item.replace("정보 더 보기", "").strip()
        print(author)

백온유, 강보라, 서장원, 성해나, 성혜령 저 외 2명


In [23]:
for item in text_clean(book_list[18].select_one(".authPub.info_auth").text).split("/"):
#     print(item)
    if "감추기" in item[:4]:
        author = item.replace("감추기", "").strip()
        print(author)

백온유 강보라 서장원 성해나 성혜령 이희주 현호정
